# NBA Future Game Predictions

This notebook generates scoring predictions for upcoming NBA games using previously trained machine learning models and engineered pre-game features.

It loads the trained model artifacts, prepares input data, generates predictions, and stores the results for further analysis.

## Prediction Workflow

1. Load trained models and feature configuration
2. Load and prepare upcoming game data
3. Generate team scoring predictions
4. Export prediction results

## 1. Generate Future Game Predictions

Loads trained model artifacts and the latest future game dataset, applies the required feature transformations, generates team scoring predictions, and appends the results to the prediction history.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib

# ================= SETTINGS =================
DATA_PATH = "future_dataset.parquet"
MODEL_PATH = "output/xboost_PTS_for.pkl"
OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)

HISTORY_PATH = f"{OUT_DIR}/future_PTS_predictions_history.csv"

# ================= LOAD DATA =================
df = pd.read_parquet(DATA_PATH)
df = df.sort_values("Date").reset_index(drop=True)

# ================= LOAD MODEL =================
bundle = joblib.load(MODEL_PATH)

xgb_model = bundle["xgb"]
feature_cols = bundle["features"]
feature_medians = bundle["feature_medians"]

print("✅ Model loaded")
print(f"N features: {len(feature_cols)}")

# ================= FEATURE PREP =================
X = df[feature_cols].copy()

for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce").astype("float32")
    X[col] = X[col].fillna(feature_medians[col])

# ================= PREDICT =================
preds_xgb = xgb_model.predict(X)

df["Pred_PTS_for"] = preds_xgb

# ================= OUTPUT (APPEND MODE) =================
out_cols = [
    "event_key",
    "Date",
    "Team",
    "Pred_PTS_for",
]

out_df = df[out_cols].copy()
out_df["Pred_PTS_for"] = out_df["Pred_PTS_for"].round(0)

# метаданные запуска (очень важно для истории)
out_df["model_name"] = "xgb_PTS_for"
out_df["model_version"] = "v1"
out_df["run_ts"] = pd.Timestamp.utcnow()

# append без перезаписи
write_header = not os.path.exists(HISTORY_PATH)

out_df.to_csv(
    HISTORY_PATH,
    mode="a",
    header=write_header,
    index=False
)

print(f"\n📈 FUTURE PTS PREDICTIONS APPENDED: {len(out_df)} rows")
print(f"📁 File: {HISTORY_PATH}")
